# Word Embeddings

**Course:** [Natural Language Processing](https://ml-viz-ruby.vercel.app/courses/nlp/02-word-embeddings)

**The idea in one sentence.** *"You shall know a word by the company it keeps"* —
represent each word as a dense vector learned so that words appearing in similar
contexts get similar vectors, turning meaning into geometry.

Two routes to the same place, both in this notebook:

- **Predictive (Word2Vec skip-gram)** — a tiny neural net predicts context words
  from a centre word; the hidden weights *become* the embeddings. Built from
  scratch with negative sampling.
- **Count-based (PPMI + SVD)** — build a word×word co-occurrence matrix, reweight
  it by pointwise mutual information, and factorise it. We use this as our
  **validation**: a completely different method should recover the *same*
  semantic geometry (king ↔ queen), and it does.

Then: the famous `king − man + woman ≈ queen` analogy, why static embeddings
break on **polysemy** (`bank`), gotchas, and exercises.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Skip-gram Word2Vec — toy implementation

Skip-gram learns embeddings by training a neural network to predict context words from center words.

In [ ]:
corpus = [
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king and queen are royalty",
    "the man went to the market",
    "the woman went to the market",
    "the man and woman are human",
    "royalty includes king and queen",
    "human includes man and woman",
]

# Build vocabulary
tokens = ' '.join(corpus).split()
vocab = sorted(set(tokens))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}
V = len(vocab)
print(f"Vocabulary ({V} words): {vocab}")

In [ ]:
# Generate skip-gram training pairs
def generate_pairs(corpus, window=2):
    pairs = []
    for sentence in corpus:
        words = sentence.split()
        for i, center in enumerate(words):
            for j in range(max(0, i-window), min(len(words), i+window+1)):
                if i != j:
                    pairs.append((w2i[center], w2i[words[j]]))
    return pairs

pairs = generate_pairs(corpus)
print(f"Generated {len(pairs)} training pairs")
print("Sample:", [(i2w[c], i2w[ctx]) for c, ctx in pairs[:5]])

In [ ]:
# Train skip-gram with negative sampling
D = 4  # embedding dimension
W = np.random.randn(V, D) * 0.1   # input embeddings (the word vectors)
C = np.random.randn(V, D) * 0.1   # output embeddings

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -20, 20)))

def train_step(center_idx, context_idx, neg_samples, lr=0.05):
    # Positive pair gradient
    pos_score = np.dot(W[center_idx], C[context_idx])
    pos_grad = (sigmoid(pos_score) - 1)
    W[center_idx] -= lr * pos_grad * C[context_idx]
    C[context_idx] -= lr * pos_grad * W[center_idx]
    # Negative pairs gradient
    for neg_idx in neg_samples:
        neg_score = np.dot(W[center_idx], C[neg_idx])
        neg_grad = sigmoid(neg_score)
        W[center_idx] -= lr * neg_grad * C[neg_idx]
        C[neg_idx] -= lr * neg_grad * W[center_idx]

# Training loop
k = 5  # negative samples
for epoch in range(200):
    np.random.shuffle(pairs)
    for center, context in pairs:
        negs = np.random.choice([i for i in range(V) if i != context], k)
        train_step(center, context, negs, lr=0.05)

print("Training complete.")
print("\nEmbedding for 'king':", W[w2i['king']].round(3))

## Semantic geometry: king − man + woman

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def most_similar(query_vec, exclude=(), top_n=3):
    sims = [(i2w[i], cosine_sim(query_vec, W[i])) for i in range(V) if i2w[i] not in exclude]
    return sorted(sims, key=lambda x: -x[1])[:top_n]

# Analogy: king - man + woman ≈ ?
analogy = W[w2i['king']] - W[w2i['man']] + W[w2i['woman']]
result = most_similar(analogy, exclude={'king', 'man', 'woman'})
print("king − man + woman ≈", result)

## The library way — validate against count-based PPMI + SVD embeddings

Skip-gram is *predictive*. The classic *count-based* alternative builds a
word×word co-occurrence matrix, reweights it by **positive pointwise mutual
information** (PPMI, so frequent-but-uninformative pairs are down-weighted), and
factorises it with truncated SVD. Levy & Goldberg (2014) showed these two
approaches are deeply related — so they should recover the **same geometry**.

We build it with `sklearn`'s `TruncatedSVD` and check that, just like our
skip-gram, `king`'s nearest neighbour is `queen` and royalty clusters away from
humans — a strong cross-method sanity check on both implementations.

In [ ]:
from sklearn.decomposition import TruncatedSVD

# 1. co-occurrence counts (window=2)
Xco = np.zeros((V, V))
for sentence in corpus:
    ws = sentence.split()
    for i, cen in enumerate(ws):
        for j in range(max(0, i-2), min(len(ws), i+3)):
            if i != j:
                Xco[w2i[cen], w2i[ws[j]]] += 1

# 2. positive PMI reweighting
total = Xco.sum()
Pxy = Xco / total
Px = Xco.sum(1, keepdims=True) / total
Py = Xco.sum(0, keepdims=True) / total
with np.errstate(divide='ignore', invalid='ignore'):
    pmi = np.log2(Pxy / (Px * Py))
pmi[~np.isfinite(pmi)] = 0.0
ppmi = np.maximum(pmi, 0.0)

# 3. factorise
E = TruncatedSVD(n_components=4, random_state=0).fit_transform(ppmi)

def nn_count(w, top=3):
    sims = [(i2w[i], cosine_sim(E[w2i[w]], E[i])) for i in range(V) if i != w2i[w]]
    return sorted(sims, key=lambda x: -x[1])[:top]

print("count-based (PPMI+SVD) nearest neighbours:")
print("  king :", nn_count('king'))
print("  man  :", nn_count('man'))
assert nn_count('king')[0][0] == 'queen', 'PPMI+SVD should also put queen nearest to king'
assert cosine_sim(E[w2i['king']], E[w2i['queen']]) > cosine_sim(E[w2i['king']], E[w2i['man']])
print("\n✅ a completely different method recovers the same king↔queen geometry as skip-gram")

In [ ]:
# PCA projection for visualization (D=4 → 2D)
from numpy.linalg import eigh

def pca2(X):
    X = X - X.mean(0)
    cov = X.T @ X / len(X)
    vals, vecs = eigh(cov)
    return X @ vecs[:, -2:]

embs_2d = pca2(W)

highlight = {'king': '#6366f1', 'queen': '#6366f1', 'man': '#2dd4bf', 'woman': '#2dd4bf',
             'royalty': '#f97316', 'human': '#f97316'}

fig, ax = plt.subplots(figsize=(8, 6))
for i, word in enumerate(vocab):
    color = highlight.get(word, '#64748b')
    ax.scatter(*embs_2d[i], color=color, s=80, zorder=3)
    ax.annotate(word, embs_2d[i], textcoords='offset points', xytext=(6, 3),
                fontsize=9, color=color)

# Draw analogy arrows
for a, b, c, d in [('man', 'king', 'woman', 'queen')]:
    va, vb = embs_2d[w2i[a]], embs_2d[w2i[b]]
    vc, vd = embs_2d[w2i[c]], embs_2d[w2i[d]]
    ax.annotate('', xy=vb, xytext=va, arrowprops=dict(arrowstyle='->', color='#6366f1', alpha=0.6))
    ax.annotate('', xy=vd, xytext=vc, arrowprops=dict(arrowstyle='->', color='#2dd4bf', alpha=0.6))

# Legend patches
legend_items = [
    mpatches.Patch(color='#6366f1', label='Royalty (king/queen)'),
    mpatches.Patch(color='#2dd4bf', label='Human (man/woman)'),
    mpatches.Patch(color='#f97316', label='Category words'),
]
ax.legend(handles=legend_items, loc='lower right', facecolor='#1a1d27', edgecolor='#2a2d3a')
ax.set_title('Word2Vec embeddings — PCA projection (king−man+woman≈queen)', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What to notice — the embedding map.** Royalty (`king`/`queen`) and humans
(`man`/`woman`) form separate clusters, and the `man→king` and `woman→queen`
arrows are roughly **parallel**: the "royalty" direction is a consistent vector
you can add to any human word. That parallelism is *exactly* what makes vector
arithmetic (`king − man + woman ≈ queen`) work.

## Static vs contextual embeddings: polysemy

Static embeddings assign one vector per word type. Contextual embeddings (like BERT) assign different vectors depending on sentence context.

In [ ]:
# Simulate static vs contextual with simple averaging (static) vs sentence context (contextual)
bank_sentences = [
    "She sat by the river bank watching ducks",
    "The river bank was covered in mud",
    "He opened a bank account for savings",
    "The bank approved his mortgage loan",
]

# For illustration: 'river/mud/ducks/watching' → nature context, 'account/savings/loan/mortgage' → finance
# Static embedding: always the same random vector for 'bank'
static_bank = np.array([0.3, 0.1, -0.2, 0.4])  # fixed for all contexts

# Contextual: different vector based on surrounding words (conceptual)
nature_context = np.array([0.8, 0.1, 0.7, 0.2])   # "river", "mud", "ducks" nearby
finance_context = np.array([-0.1, 0.9, -0.3, 0.8])  # "account", "loan" nearby

contextual_bank = [
    nature_context, nature_context,
    finance_context, finance_context,
]

print("Static 'bank' vector (same for all sentences):")
print(f"  {static_bank}")
print("\nContextual 'bank' vectors:")
for i, s in enumerate(bank_sentences):
    print(f"  Sentence {i+1}: {contextual_bank[i]}  | '{s[:45]}...'")

In [ ]:
# Similarity: river-bank with finance-bank
print("Static: cosine(bank_s1, bank_s3) =", cosine_sim(static_bank, static_bank).round(3),
      "(always 1.0 — same vector)")
print("Contextual: cosine(bank_nature, bank_finance) =",
      cosine_sim(nature_context, finance_context).round(3),
      "(different senses → low similarity)")

**What to notice — polysemy.** A *static* embedding gives `bank` one vector, so
"river bank" and "savings bank" are forced to cosine 1.0 — the model literally
cannot tell the senses apart. *Contextual* embeddings (BERT, next lesson) read
the surrounding words and emit a different vector per sense, which is why they
replaced static vectors for almost everything.

## Gotchas & tradeoffs

| Issue | Why it bites | Mitigation |
|-------|--------------|------------|
| **static = one vector per type** | can't handle polysemy (`bank`, `plant`, `bat`) | contextual embeddings (BERT) |
| **negative sampling count $k$** | too few → noisy; too many → slow | $k=5$–$20$ (small data), $2$–$5$ (large) |
| **frequency bias** | frequent words dominate co-occurrence | PPMI / subsampling frequent words |
| **analogy over-hyped** | `king−man+woman` works on curated sets, fails on messy ones | treat as a *sanity check*, not a benchmark |
| **cosine, not Euclidean** | vector *magnitude* encodes frequency, not meaning | compare with cosine similarity |

Demo: PMI beats raw counts at telling *association* from mere *frequency* — a
common word co-occurs with everything, so raw counts overstate its links.

In [ ]:
# 'the' is frequent, so it has HIGH raw co-occurrence with 'king' — but that's just
# frequency, not association. PMI corrects for the marginals and deflates it.
def raw_and_pmi(a, b):
    return Xco[w2i[a], w2i[b]], ppmi[w2i[a], w2i[b]]

for pair in [('king', 'the'), ('king', 'queen')]:
    raw, p = raw_and_pmi(*pair)
    print(f'{pair[0]:>5} <-> {pair[1]:<6}  raw count = {raw:.0f}   PPMI = {p:.3f}')
print("\n'the' co-occurs with 'king' by raw count, but PPMI ~ 0 (it co-occurs with EVERYTHING).")
print("'queen' has a lower/comparable raw count yet high PPMI — a genuine association.")

## ✏️ Your turn

### Exercise 1: Implement cosine-similarity analogy solver

Given embeddings for a, b, c, find the word d such that `a : b :: c : d` (i.e., `b - a + c ≈ d`).

In [ ]:
def solve_analogy(a, b, c, embeddings, vocab_list):
    """
    Solve analogy: a is to b as c is to ?
    Returns the word with highest cosine similarity to (b - a + c),
    excluding a, b, c themselves.
    
    Args:
        a, b, c: str, word strings
        embeddings: dict mapping word -> np.array
        vocab_list: list of all word strings
    Returns:
        str: best matching word
    """
    # TODO(you): compute query vector, find nearest neighbor excluding a, b, c
    pass


emb_dict = {w: W[w2i[w]] for w in vocab}
answer = solve_analogy('man', 'king', 'woman', emb_dict, vocab)
print(f"man : king :: woman : {answer}")

In [ ]:
answer = solve_analogy('man', 'king', 'woman', emb_dict, vocab)
assert isinstance(answer, str), "Should return a string"
assert answer not in ('man', 'king', 'woman'), "Should not return the input words"

# Edge case: a tiny 4-word vocabulary where only one candidate remains after
# excluding a, b, c — the solver must still pick it (no off-by-one exclusion bug).
tiny_vocab = ['a', 'b', 'c', 'd']
tiny_emb = {
    'a': np.array([1.0, 0.0]),
    'b': np.array([0.0, 1.0]),
    'c': np.array([1.0, 1.0]),
    'd': np.array([-1.0, -1.0]),
}
tiny_answer = solve_analogy('a', 'b', 'c', tiny_emb, tiny_vocab)
assert tiny_answer == 'd', "With only one candidate left after exclusion, it must be picked"

print(f"✅ Exercise 1 passed — analogy answer: '{answer}'")

<details>
<summary>💡 Show solution</summary>

```python
def solve_analogy(a, b, c, embeddings, vocab_list):
    query = embeddings[b] - embeddings[a] + embeddings[c]
    exclude = {a, b, c}
    best_word, best_sim = None, -float('inf')
    for word in vocab_list:
        if word in exclude:
            continue
        sim = cosine_sim(query, embeddings[word])
        if sim > best_sim:
            best_sim = sim
            best_word = word
    return best_word
```
</details>

### Exercise 2: Compute a co-occurrence matrix (GloVe style)

GloVe builds a global co-occurrence matrix X where X[i][j] counts how often word j appears in the context window of word i.

In [ ]:
def build_cooccurrence(corpus_sentences, word2idx, window=2):
    """
    Build a co-occurrence count matrix.
    
    Args:
        corpus_sentences: list of str
        word2idx: dict mapping word -> int index
        window: int, context window radius
    Returns:
        np.ndarray of shape (V, V) with co-occurrence counts
    """
    V = len(word2idx)
    X = np.zeros((V, V))
    # TODO(you): for each sentence, for each center word at position i,
    # increment X[center_idx][context_idx] for all context positions within window
    return X


X = build_cooccurrence(corpus, w2i)
print(f"Co-occurrence matrix shape: {X.shape}")
print(f"king <-> queen co-occurrences: {X[w2i['king'], w2i['queen']]:.0f}")
print(f"king <-> man co-occurrences:   {X[w2i['king'], w2i['man']]:.0f}")

In [ ]:
X = build_cooccurrence(corpus, w2i)
assert X.shape == (V, V), "Wrong shape"
assert X[w2i['king'], w2i['queen']] > 0, "king and queen should co-occur"
assert X[w2i['king'], w2i['man']] >= 0, "Should have non-negative counts"
assert X[w2i['king'], w2i['king']] == 0 or True, "Self co-occurrence can be 0"

# Edge case: a single-word "sentence" has no neighbors, so every count stays 0 —
# the function must not crash or index out of range on a length-1 sentence.
single_word_corpus = ["kingdom"]
X_single = build_cooccurrence(single_word_corpus, w2i)
assert X_single.shape == (V, V), "Shape should still be (V, V) regardless of corpus size"
assert X_single.sum() == 0, "A lone word has no context, so all co-occurrence counts must be 0"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def build_cooccurrence(corpus_sentences, word2idx, window=2):
    V = len(word2idx)
    X = np.zeros((V, V))
    for sentence in corpus_sentences:
        words = sentence.split()
        for i, center in enumerate(words):
            if center not in word2idx:
                continue
            ci = word2idx[center]
            for j in range(max(0, i-window), min(len(words), i+window+1)):
                if i != j and words[j] in word2idx:
                    X[ci][word2idx[words[j]]] += 1
    return X
```
</details>

---
### Extra practice — DML #111: Pointwise Mutual Information

Raw co-occurrence counts (Exercise 2 above) can't distinguish "these words co-occur a lot because they're related" from "these words co-occur a lot because both are simply frequent." The [Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) **PMI** formulation corrects for that by comparing the observed joint probability to what independence would predict:

$$
\mathrm{PMI}(x, y) = \log_2 \frac{P(x, y)}{P(x)\,P(y)}
$$

This is exactly the intuition GloVe is built on — factorizing a matrix of PMI-like values instead of raw counts, since raw counts are dominated by word frequency rather than semantic association.

In [ ]:
def compute_pmi(joint_counts, total_counts_x, total_counts_y, total_samples):
    """
    DML #111 -- Pointwise Mutual Information from raw counts.

    Args:
        joint_counts: number of samples where x and y both occur
        total_counts_x, total_counts_y: marginal occurrence counts of x, y
        total_samples: total number of samples the counts are drawn from
    Returns:
        float, PMI in bits (log base 2), rounded to 3 decimals;
        -inf if x and y never co-occur.
    """
    p_x = total_counts_x / total_samples
    p_y = total_counts_y / total_samples

    # TODO(you): joint probability P(x, y)
    p_xy = ...

    if p_xy == 0:
        return float('-inf')

    # TODO(you): log2(P(x,y) / (P(x) * P(y))), rounded to 3 decimals
    pmi = ...
    return pmi


print(compute_pmi(50, 200, 300, 1000))

In [ ]:
# Checks — run me (DML's own test cases)
assert compute_pmi(10, 50, 50, 200) == -0.322
assert compute_pmi(100, 500, 500, 1000) == -1.322
assert compute_pmi(50, 200, 300, 1000) == -0.263

# Edge case: events that never co-occur -> PMI is -inf ("maximally surprising" absence)
assert compute_pmi(0, 50, 50, 200) == float('-inf')

# Edge case: statistically independent events (p_xy == p_x * p_y exactly) -> PMI == 0
assert compute_pmi(1, 10, 10, 100) == 0.0

print("✅ DML #111 passed")

<details>
<summary>💡 Show solution</summary>

```python
def compute_pmi(joint_counts, total_counts_x, total_counts_y, total_samples):
    p_x = total_counts_x / total_samples
    p_y = total_counts_y / total_samples
    p_xy = joint_counts / total_samples
    if p_xy == 0:
        return float('-inf')
    pmi = np.log2(p_xy / (p_x * p_y))
    return round(float(pmi), 3)
```
</details>

## Key takeaways

- **Meaning becomes geometry.** Words in similar contexts get similar vectors;
  semantic relationships become consistent *directions* (the royalty vector).
- **Two routes, one destination.** Predictive skip-gram and count-based PPMI+SVD
  recover the same king↔queen structure — we verified it here.
- **Reweight your counts.** Raw co-occurrence is dominated by frequency; PMI/PPMI
  isolates genuine association, which is what GloVe factorises.
- **Static embeddings can't do polysemy.** One vector per word type forces
  `bank`'s senses together — the motivation for contextual embeddings (next lesson).
- **Compare with cosine, and treat analogies as a sanity check**, not a benchmark.